# VINDHYA — राष्ट्रीय जलवायु गणना (Colab)

**कैसे चलाना है**

1. बाईं तरफ़ 🔑 (key) icon → दो secret जोड़िए, दोनों का toggle चालू कीजिए:
   - `GH_TOKEN` — GitHub का fine-grained token (Contents: Read and write)
   - `GEE_KEY` — service account JSON की पूरी सामग्री
2. ऊपर **Runtime → Run all**
3. Session कट जाए तो यही notebook दोबारा खोलिए और फिर **Run all** — जहाँ से कटा, वहीं से चलेगा

**यह notebook आपके लैपटॉप पर कुछ नहीं उतारता।** सब Google के सर्वर पर होता है।

---

**डेटा कहाँ से आता है**

| क्या | कहाँ से | टिप्पणी |
|---|---|---|
| Tmax / Tmin / वर्षा | पहले `imdlib` (असली IMD), न चले तो GEE का ERA5-Land + CHIRPS | notebook ख़ुद बता देगा कौन सा चला |
| NDVI, LST, मिट्टी की नमी | GEE | Google के सर्वर पर गणना |
| 2050 projection | GEE — NEX-GDDP-CMIP6 | observed से हमेशा अलग |
| सीमाएँ | Survey of India, repo में पहले से | |

कोई भी मान गढ़ा नहीं जाएगा। जहाँ डेटा नहीं, वहाँ `data not available` लिखा जाएगा।

## 1. मशीन की जाँच

In [ ]:
import subprocess, sys, platform
print("Python :", sys.version.split()[0], "|", platform.platform())
print()
print(subprocess.run(["df","-h","/content"],capture_output=True,text=True).stdout)
print(subprocess.run(["free","-h"],capture_output=True,text=True).stdout)
print(subprocess.run(["nproc"],capture_output=True,text=True).stdout.strip(), "CPU cores")

## 2. Packages

In [ ]:
%%capture --no-stderr
!pip install -q earthengine-api geopandas xarray netCDF4 rioxarray scipy tqdm imdlib

In [ ]:
for m in ("ee","geopandas","xarray","scipy","tqdm"):
    try:
        __import__(m); print(f"  ok    {m}")
    except Exception as e:
        print(f"  FAIL  {m}: {e}")
try:
    import imdlib; print("  ok    imdlib (asli IMD data mil sakta hai)")
except Exception as e:
    print(f"  ---   imdlib nahi mila: {e}")

## 3. Google Drive — प्रगति यहाँ सुरक्षित रहेगी

Session कटने पर `/content` मिट जाता है, Drive नहीं। इसीलिए progress यहाँ रखी जाती है।

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

WORK = Path('/content/drive/MyDrive/vindhya')
WORK.mkdir(parents=True, exist_ok=True)
PROGRESS = WORK / 'progress.json'
print("Drive taiyaar :", WORK)
print("Progress file :", PROGRESS, "(maujood)" if PROGRESS.exists() else "(pehli baar)")

## 4. Secrets

Token और key यहीं से आते हैं — notebook में कभी नहीं लिखे जाते।

In [ ]:
from google.colab import userdata
import json, os, sys

def secret(name):
    try:
        v = userdata.get(name)
    except Exception as e:
        sys.exit(f"STOP: secret '{name}' nahi mila.\n"
                 f"  Bayen taraf key icon -> Add new secret -> naam '{name}' -> toggle chalu.\n  ({e})")
    if not v or not v.strip():
        sys.exit(f"STOP: secret '{name}' khali hai.")
    return v.strip()

GH_TOKEN = secret('GH_TOKEN')
GEE_KEY_RAW = secret('GEE_KEY')

try:
    gee_info = json.loads(GEE_KEY_RAW)
    SA_EMAIL = gee_info['client_email']
    GEE_PROJECT = gee_info['project_id']
except Exception as e:
    sys.exit(f"STOP: GEE_KEY sahi JSON nahi hai: {e}\n"
             f"  Mac par chalao: cat ~/.gee/service-account.json\n"
             f"  Poora text copy karke secret me chipkao.")

KEYFILE = '/content/gee_key.json'          # /content me, repo me KABHI nahi
with open(KEYFILE,'w') as f: f.write(GEE_KEY_RAW)
os.chmod(KEYFILE, 0o600)

print("GitHub token : mil gaya")
print("GEE account  :", SA_EMAIL)
print("GEE project  :", GEE_PROJECT)

## 5. Repo

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path('/content/vindhya-climate-portal')
URL  = f"https://{GH_TOKEN}@github.com/vindhyaresearch25-a11y/vindhya-climate-portal.git"

if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'pull','--rebase'], check=True)
else:
    subprocess.run(['git','clone','--depth','1',URL,str(REPO)], check=True)

subprocess.run(['git','-C',str(REPO),'config','user.name','vindhya-colab-bot'], check=True)
subprocess.run(['git','-C',str(REPO),'config','user.email','actions@github.com'], check=True)
os.chdir(REPO)

print("Repo :", REPO)
print(subprocess.run(['git','log','--oneline','-3'],capture_output=True,text=True).stdout)

## 6. GEE चालू है या नहीं — असली query से जाँच

यहाँ फेल हुआ तो आगे मत बढ़िए। ख़ाली नतीजा डेटा जैसा दिखता है, वही सबसे ख़तरनाक है।

In [ ]:
import ee, sys

try:
    creds = ee.ServiceAccountCredentials(SA_EMAIL, KEYFILE)
    ee.Initialize(creds, project=GEE_PROJECT)
except Exception as e:
    sys.exit(f"STOP: Earth Engine ne credentials nahi maane.\n  {e}\n\n"
             f"  Teen me se koi ek adhoora hai:\n"
             f"   1. Cloud project Earth Engine ke liye register nahi\n"
             f"      https://code.earthengine.google.com/register\n"
             f"   2. Earth Engine API enabled nahi\n"
             f"   3. SERVICE ACCOUNT khud register nahi: {SA_EMAIL}\n"
             f"      (yahi step log bhoolte hain, 403 isi se aata hai)")

pt = ee.Geometry.Point([77.413, 23.260])          # Bhopal
n_era = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
          .filterDate('2024-01-01','2024-02-01').size().getInfo()
n_chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
             .filterBounds(pt).filterDate('2024-01-01','2024-02-01').size().getInfo()

print("Earth Engine chalu.")
print(f"  ERA5-Land daily images (Jan 2024) : {n_era}")
print(f"  CHIRPS daily images (Jan 2024)    : {n_chirps}")
if n_era == 0 or n_chirps == 0:
    raise SystemExit("STOP: authentication to chali, par data query khali aayi. Aage mat badho.")

## 7. असली IMD मिल सकता है क्या?

`imdlib` सीधे IMD Pune से उतारता है। चल गया तो ERA5 की जगह असली IMD मिलेगा — जो कहीं बेहतर है, क्योंकि पोर्टल हर जगह IMD लिखता है।

**यह cell फेल हो जाए तो कोई दिक़्क़त नहीं** — नीचे GEE वाला रास्ता चलता रहेगा। बस नतीजा मुझे बता दीजिए।

In [ ]:
IMD_AVAILABLE = False
IMD_NOTE = ""
try:
    import imdlib as imd, time
    t0 = time.time()
    d = imd.get_data('tmax', 2020, 2020, fn_format='yearwise')
    dt = time.time() - t0
    IMD_AVAILABLE = True
    IMD_NOTE = f"imdlib chala: tmax 2020 utar gaya, {dt:.0f} sec"
    print("ASLI IMD MIL GAYA.")
    print(" ", IMD_NOTE)
    print("  ->", d)
    print("\n  YE NATEEJA MUJHE BHEJIYE. Isse tay hoga ki ERA5 ki jagah")
    print("     asli IMD istemal ho sakta hai ya nahi.")
except Exception as e:
    IMD_NOTE = f"imdlib nahi chala: {type(e).__name__}: {e}"
    print("imdlib se IMD nahi mila.")
    print(" ", IMD_NOTE)
    print("\n  Koi dikkat nahi -- neeche GEE (ERA5-Land + CHIRPS) wala")
    print("     rasta chalega. Har output file me saaf likha jayega ki")
    print("     data ERA5/CHIRPS ka hai, IMD ka NAHI.")
    print("\n  YE MESSAGE MUJHE BHEJIYE.")

## 8. Connectivity और लागत की जाँच (repo की अपनी script)

In [ ]:
!python scripts/08_gee_national_climate.py --stage validate

## 9. Madhya Pradesh — पहला benchmark

पहले सिर्फ़ MP। असली समय और आकार यहीं से पता चलेगा, अंदाज़े से नहीं।

In [ ]:
import time, subprocess
from pathlib import Path

def dir_size_mb(p):
    p = Path(p)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())/1e6 if p.exists() else 0.0

OUT = Path('dashboard/data/climate')
before_n, before_mb = len(list(OUT.rglob('*.json'))), dir_size_mb(OUT)

t0 = time.time()
r = subprocess.run(['python','scripts/08_gee_national_climate.py',
                    '--stage','run','--states','MP','--resume'],
                   text=True)
mins = (time.time()-t0)/60

after_n, after_mb = len(list(OUT.rglob('*.json'))), dir_size_mb(OUT)
new_files = after_n - before_n

print("\n" + "="*58)
print("MADHYA PRADESH BENCHMARK")
print("="*58)
print(f"  exit code        : {r.returncode}")
print(f"  samay            : {mins:.1f} minute")
print(f"  nayi files       : {new_files}")
print(f"  kul climate JSON : {after_n}  ({after_mb:.1f} MB)")
if new_files > 0:
    per = mins/new_files
    print(f"  prati zila       : {per:.2f} min, {(after_mb-before_mb)/new_files:.2f} MB")
    print(f"  733 zilon ka ANUMAN : {per*733/60:.1f} ghante, "
          f"{(after_mb-before_mb)/new_files*733:.0f} MB")
print("="*58)
print("\nYE POORA BLOCK MUJHE BHEJIYE.")

## 10. GitHub पर भेजिए

In [ ]:
import subprocess

def push(msg):
    subprocess.run(['git','add','-A'], check=True)
    st = subprocess.run(['git','diff','--staged','--quiet'])
    if st.returncode == 0:
        print("Kuch naya nahi -- push ki zaroorat nahi.")
        return
    subprocess.run(['git','commit','-m',msg], check=True)
    subprocess.run(['git','pull','--rebase'], check=True)
    subprocess.run(['git','push'], check=True)
    print("Push ho gaya:", msg)

push("data(climate): Madhya Pradesh benchmark from Colab")

## 11. पूरा देश — resume के साथ

यह cell राज्य-दर-राज्य चलती है और हर राज्य के बाद push करती है।

**Session कट जाए तो notebook दोबारा खोलकर Run all दबाइए** — जो राज्य हो चुके, वे छोड़ दिए जाएँगे।

In [ ]:
import json, time, subprocess
from pathlib import Path
from datetime import datetime, timezone, timedelta

IST = timezone(timedelta(hours=5, minutes=30))

# --- progress Drive me, taaki session katne par bhi bacha rahe ---
prog = json.loads(PROGRESS.read_text()) if PROGRESS.exists() else {"done": [], "log": []}
done = set(prog["done"])
print(f"Ab tak poore hue rajya: {len(done)}")
if done: print("  ", ", ".join(sorted(done)))

# --- rajya ki suchi ASLI boundary folder se, kabhi type karke nahi ---
VILL = Path('dashboard/data/boundaries/soi/villages')
if not VILL.exists():
    raise SystemExit(f"STOP: {VILL} nahi mila. Boundary layer pehle chahiye.")
STATES = sorted(p.name for p in VILL.iterdir() if p.is_dir())
print(f"Kul rajya/UT: {len(STATES)}")

pending = [s for s in STATES if s not in done]
print(f"Baaki: {len(pending)}\n")

OUT = Path('dashboard/data/climate')

for i, state in enumerate(pending, 1):
    print(f"\n[{i}/{len(pending)}] {state}")
    t0 = time.time()
    n0, mb0 = len(list(OUT.rglob('*.json'))), dir_size_mb(OUT)

    r = subprocess.run(['python','scripts/08_gee_national_climate.py',
                        '--stage','run','--states',state,'--resume'], text=True)

    mins = (time.time()-t0)/60
    n1, mb1 = len(list(OUT.rglob('*.json'))), dir_size_mb(OUT)

    entry = {
        "state": state,
        "exit_code": r.returncode,
        "minutes": round(mins,1),
        "new_districts": n1-n0,
        "mb_added": round(mb1-mb0,2),
        "finished_at": datetime.now(IST).isoformat(timespec='seconds'),
    }
    prog["log"].append(entry)
    print("   ", entry)

    if r.returncode != 0:
        PROGRESS.write_text(json.dumps(prog, indent=1))
        raise SystemExit(f"STOP: {state} fail hua (exit {r.returncode}). "
                         f"Aage nahi badh rahe. Ye poora message bhejiye.")

    try:
        push(f"data(climate): {state} [skip ci]")
    except subprocess.CalledProcessError as e:
        PROGRESS.write_text(json.dumps(prog, indent=1))
        raise SystemExit(f"STOP: push fail hua {state} ke baad: {e}")

    prog["done"].append(state)
    done.add(state)
    PROGRESS.write_text(json.dumps(prog, indent=1))

print("\n" + "="*58)
print(f"POORE HO GAYE: {len(prog['done'])} / {len(STATES)} rajya")
print("="*58)

## 12. सारांश — यह मुझे भेजिए

In [ ]:
import json
from pathlib import Path

prog = json.loads(PROGRESS.read_text()) if PROGRESS.exists() else {"done": [], "log": []}
OUT = Path('dashboard/data/climate')
files = list(OUT.rglob('*.json'))
mb = sum(f.stat().st_size for f in files)/1e6

print("="*58)
print("VINDHYA -- COLAB RUN SUMMARY")
print("="*58)
print(f"Data source     : {'ASLI IMD (imdlib)' if IMD_AVAILABLE else 'GEE ERA5-Land + CHIRPS (IMD NAHI)'}")
print(f"  {IMD_NOTE}")
print(f"Rajya poore     : {len(prog['done'])}")
print(f"District JSON   : {len(files)}")
print(f"Kul size        : {mb:.1f} MB")
if prog['log']:
    tot = sum(e['minutes'] for e in prog['log'])
    print(f"Kul samay       : {tot/60:.1f} ghante")
    print("\nRajya-war:")
    for e in prog['log']:
        print(f"  {e['state']:<24} {e['minutes']:>7.1f} min  "
              f"{e['new_districts']:>4} zile  {e['mb_added']:>7.2f} MB")
print("="*58)
print("\nYE POORA BLOCK MUJHE BHEJIYE.")